# Week 6: CUDA video preprocessing pipeline

Select **Runtime → Change runtime type → T4 GPU**. Push the Week 6 source commit first; this notebook clones the repository from GitHub. It builds and runs the CPU fallback and CUDA tests, then benchmarks one stream against two streams with pinned host buffers. The measured input is synthetic, already-decoded RGB frames; it does not include video decoding or detector inference. Raw results are saved to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive")
REPO_URL = "https://github.com/elgunismayil0v2001/vision-engine.git"
REPO_DIR = Path("/content/vision-engine")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / "cpp/src/video_pipeline_cuda.cu").is_file():
    raise FileNotFoundError("Push the Week 6 commit before running this notebook.")
BUILD_DIR = Path("/content/vision-engine-build/cpp-week6-cuda")
DRIVE_OUT = Path("/content/drive/MyDrive/vision-engine-results")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
LOG_PATH = DRIVE_OUT / "week6_t4.json"
print("Repository:", REPO_DIR)
print("Raw result:", LOG_PATH)

In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select a Colab T4 runtime.")
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

In [ ]:
import importlib.metadata
import sys

required = {"onnx": "1.20.1", "onnxruntime": "1.31.1"}
needs_install = False
for package, version in required.items():
    try:
        needs_install |= importlib.metadata.version(package) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    specs = [f"{name}=={version}" for name, version in required.items()]
    subprocess.run([sys.executable, "-m", "pip", "install", *specs], check=True)
ORT_VERSION = "1.31.1"
ORT_ROOT = Path("/content/onnxruntime-sdk")
if not (ORT_ROOT / "lib/libonnxruntime.so").is_file():
    archive = Path("/content/onnxruntime-linux-x64.tgz")
    url = f"https://github.com/microsoft/onnxruntime/releases/download/v{ORT_VERSION}/onnxruntime-linux-x64-{ORT_VERSION}.tgz"
    subprocess.run(["curl", "-fL", url, "-o", str(archive)], check=True)
    ORT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(archive), "--strip-components=1", "-C", str(ORT_ROOT)], check=True)

In [ ]:
configure = [
    "cmake", "-S", str(REPO_DIR / "cpp"), "-B", str(BUILD_DIR),
    f"-DONNXRUNTIME_ROOT={ORT_ROOT}", f"-DPython3_EXECUTABLE={sys.executable}",
    "-DENABLE_CUDA=ON", "-DENABLE_OPENMP=ON",
    "-DCMAKE_CUDA_ARCHITECTURES=75", "-DBUILD_TESTING=ON",
]
subprocess.run(configure, check=True)
subprocess.run(["cmake", "--build", str(BUILD_DIR), "-j2"], check=True)
subprocess.run(["ctest", "--test-dir", str(BUILD_DIR), "--output-on-failure"], check=True)

## Benchmark stream overlap

The benchmark warms both configurations, then processes 120 deterministic 640×480 RGB frames. It reports FPS and p50/p99/p99.9 for host staging, H2D, normalization, D2H, output copy, and per-frame end-to-end latency. CUDA timings use events; FPS includes CPU staging and output copies.

In [ ]:
import json
benchmark = [
    sys.executable, str(REPO_DIR / "bench/video/run_video_benchmark.py"),
    "--binary", str(BUILD_DIR / "video_pipeline_benchmark"),
    "--output", str(LOG_PATH), "--width", "640", "--height", "480",
    "--frames", "120", "--warmup-frames", "16",
]
subprocess.run(benchmark, cwd=REPO_DIR, check=True)
result = json.loads(LOG_PATH.read_text())
if not all(row["cuda_active"] for row in result["results"]):
    raise RuntimeError("This run used CPU fallback; no CUDA FPS was measured.")
for row in result["results"]:
    print(f"{row['stream_count']} stream(s): {row['fps']:.1f} FPS; "
          f"gain={row['fps_gain_vs_single_stream']:.2f}x; "
          f"end-to-end p99={row['stages_ms']['end_to_end']['p99_ms']:.3f} ms")
print("Saved:", LOG_PATH)